In [1]:
! pip install langchain
! pip install langchain-neo4j langchain-groq

In [47]:
import os
import unicodedata
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

load_dotenv()

# 1. Conectar o LangChain ao seu Neo4j local
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

# 2. Inicializar o "Cérebro"
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0 
)

# 3. O Prompt Turbinado com APOC e Regras de Negócio
cypher_template = """Você é um especialista em banco de dados Neo4j traduzindo perguntas em português para queries Cypher.

REGRAS OBRIGATÓRIAS:
1. Use APENAS os nós e relacionamentos fornecidos no schema.
2. NUNCA faça buscas textuais exatas usando '='. 
3. SEMPRE use a função apoc.text.clean() EM AMBOS OS LADOS da comparação com CONTAINS. O APOC remove acentos e espaços.
   Exemplo exigido: apoc.text.clean(e.nome) CONTAINS apoc.text.clean('itau cartoes')
4. CONTEXTO DE NEGÓCIO DA BASE DE DADOS:
   - Marcas, bancos, empresas e aplicativos (ex: Itaú Cartões, Uber, iFood, AmoraDev) ficam SEMPRE no nó Estabelecimento (e.nome).
   - Setores amplos, tipos de despesas e receitas (ex: Transporte, Alimentação, Juros, Salário) ficam SEMPRE no nó Categoria (cat.nome).
   - Transações de saída de dinheiro têm a propriedade t.tipo = 'Saída'. Transações de ganho de dinheiro têm t.tipo = 'Entrada'.
5. Retorne APENAS o código Cypher válido, sem explicações.

Schema:
{schema}

Pergunta do usuário: {question}
Query Cypher:"""

cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template=cypher_template
)

# 4. Criar o Agente
chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    cypher_prompt=cypher_prompt,
    verbose=True, 
    allow_dangerous_requests=True 
)

# 5. O Escudo Python
def limpar_texto(texto):
    texto_sem_acento = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    return texto_sem_acento.lower()

# 6. O Teste de Fogo Definitivo
pergunta_crua = "Qual foi o valor total que o cliente gastou com Netflix?"
pergunta = limpar_texto(pergunta_crua)
print(f"👤 Usuário: {pergunta}\n")

# Invocando o Agente
resposta = chain.invoke({"query": pergunta})

print(f"\n🤖 Resposta da IA: {resposta['result']}")

Usuário: qual foi o valor total que o cliente gastou com netflix?



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Cliente)-[:REALIZOU]->(t:Transacao)-[:NO_ESTABELECIMENTO]->(e:Estabelecimento)
WHERE t.tipo = 'Saída' AND apoc.text.clean(e.nome) CONTAINS apoc.text.clean('netflix')
RETURN sum(t.valor) AS total_gasto;
Full Context:
[{'total_gasto': 436.6}]

> Finished chain.

Resposta da IA: 436.6
